# 02 — Data Quality: all FCC technology files

Notebook 01 profiled only the **Fiber to the Premises** file (technology code 50).
This notebook validates all eight Vermont files before they are combined.

> **Naming note.** The `50` in `bdc_50_...` file names is the **state FIPS code for Vermont**, not a technology code.
> Fiber's technology code also happens to be 50, which caused early confusion: the first discovery figures
> (22 providers, ~91% single-provider locations) describe **fiber only**, not the whole broadband market.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))  # not needed when installed with `uv sync`

import pandas as pd
import matplotlib.pyplot as plt
from vermont_broadband_analysis import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
ANALYSIS = config.PROCESSED_DIR / "analysis"

## 1. File inventory
Generated by `vermont-broadband-analysis profile` (`src/vermont_broadband_analysis/fcc.py`).

In [2]:
from vermont_broadband_analysis import fcc

inventory = fcc.profile()

Profiling: bdc_50_Cable_fixed_broadband_D25_03sep2026.csv
Profiling: bdc_50_Copper_fixed_broadband_D25_03sep2026.csv


Profiling: bdc_50_FibertothePremises_fixed_broadband_D25_18aug2026.csv
Profiling: bdc_50_GSOSatellite_fixed_broadband_D25_03sep2026.csv


Profiling: bdc_50_LBRFixedWireless_fixed_broadband_D25_03sep2026.csv
Profiling: bdc_50_LicensedFixedWireless_fixed_broadband_D25_03sep2026.csv


Profiling: bdc_50_NGSOSatellite_fixed_broadband_D25_03sep2026.csv
Profiling: bdc_50_UnlicensedFixedWireless_fixed_broadband_D25_03sep2026.csv
                                                       file_name   rows technology_codes states  providers  brands  unique_locations  full_duplicates  missing_values  candidate_key_duplicates  location_provider_tech_repeats low_latency_values business_residential_codes  invalid_block_geoid_length  non_vermont_block_geoids  max_download_mbps  min_download_mbps
                  bdc_50_Cable_fixed_broadband_D25_03sep2026.csv 191040             [40]   [VT]          4       4            190998                0               0                         0                               0                [1]                     [B, X]                           0                         0               2000                 50
                 bdc_50_Copper_fixed_broadband_D25_03sep2026.csv 108650             [10]   [VT]          9       9            108552  

In [3]:
inventory[["file_name", "rows", "unique_locations", "providers", "technology_codes",
           "business_residential_codes", "candidate_key_duplicates", "location_provider_tech_repeats",
           "missing_values", "non_vermont_block_geoids"]]

,file_name,rows,unique_locations,providers,technology_codes,business_residential_codes,candidate_key_duplicates,location_provider_tech_repeats,missing_values,non_vermont_block_geoids
0,bdc_50_Cable_fixed_broadband_D25_03sep2026.csv,191040,190998,4,[40],"[B, X]",0,0,0,0
1,bdc_50_Copper_fixed_broadband_D25_03sep2026.csv,108650,108552,9,[10],"[B, R, X]",0,0,0,0
2,bdc_50_FibertothePremises_fixed_broadband_D25_...,203946,187037,22,[50],"[B, R, X]",0,0,0,0
3,bdc_50_GSOSatellite_fixed_broadband_D25_03sep2...,768584,282371,2,[60],"[B, R, X]",0,203881,0,0
4,bdc_50_LBRFixedWireless_fixed_broadband_D25_03...,7990,7982,5,[72],"[B, R, X]",0,0,0,0
5,bdc_50_LicensedFixedWireless_fixed_broadband_D...,563775,269839,5,[71],"[B, R, X]",0,17238,0,0
6,bdc_50_NGSOSatellite_fixed_broadband_D25_03sep...,282226,282226,1,[61],[X],0,0,0,0
7,bdc_50_UnlicensedFixedWireless_fixed_broadband...,9014,9009,4,[70],"[R, X]",0,0,0,0


## 2. Checks that pass in every file
* Identical 12-column schema, `state_usps = VT`, no missing values, no full-row duplicates.
* Every `block_geoid` is 15 digits and starts with `50` (Vermont).
* **Grain:** one row per `location_id + provider_id + technology + business_residential_code` (0 duplicates).

In [4]:
checks = inventory[["file_name", "missing_values", "full_duplicates", "candidate_key_duplicates",
                    "invalid_block_geoid_length", "non_vermont_block_geoids"]].set_index("file_name")
assert (checks == 0).all().all()
assert inventory["states"].map(lambda s: s == ["VT"] or s == "['VT']").all()
"All structural checks passed"

'All structural checks passed'

## 3. Anomalies and how they are handled

In [5]:
df = pd.read_parquet(config.FCC_COMBINED_PARQUET) if config.FCC_COMBINED_PARQUET.exists() else fcc.build()
len(df)

2135225

### 3.1 Same location/provider/technology listed twice (B and R)
GSO satellite (Viasat) and licensed fixed wireless (AT&T, Verizon) list some locations once as business (`B`) and once as residential (`R`), sometimes with different speeds.
**Handling:** the residential analysis keeps only `R`/`X` rows, so each location keeps its residential offer.

In [6]:
rep = df[df.duplicated(["location_id", "provider_id", "technology"], keep=False)]
rep.groupby(["technology_name", "brand_name", "business_residential_code"]).size().unstack(fill_value=0)

business_residential_code                B       R
technology_name         brand_name                
GSO Satellite           Viasat Inc  203881  203881
Licensed Fixed Wireless AT&T         13078   13078
                        Verizon       4160    4160

### 3.2 Advertised speeds of 0 Mbps
Several providers report `0/0` Mbps (mostly licensed fixed wireless: T-Mobile, VTel, AT&T, plus Consolidated copper).
The public file stores whole Mbps, so these appear to be offers below 1 Mbps.
**Handling:** kept as reported; they never meet the 25/3 or 100/20 thresholds.

In [7]:
zero = df[df["max_advertised_download_speed"] == 0]
zero.groupby(["technology_name", "brand_name"]).size().sort_values(ascending=False).rename("records").to_frame()

records
technology_name           brand_name                          
Licensed Fixed Wireless   T-Mobile                       77760
                          VTel Wireless Inc              72590
Copper                    Consolidated Communications    26180
Licensed Fixed Wireless   AT&T                           13397
Copper                    Topsham Telephone Company        814
Unlicensed Fixed Wireless New England Wireless Co          536
Copper                    GoNetspeed                       436
LBR Fixed Wireless        New England Wireless Co          329
                          NEW Alliance                     272
Unlicensed Fixed Wireless Phoenix Communications II        203
Copper                    TDS Telecom                       75
                          GREEN MOUNTAIN ACCESS              3

### 3.3 Extreme speeds
Two FirstLight Fiber records advertise 100,000 Mbps (100 Gbps). Plausible enterprise offers; they don't affect thresholds or medians.

In [8]:
df.nlargest(5, "max_advertised_download_speed")[["brand_name", "technology_name", "max_advertised_download_speed",
                                                  "max_advertised_upload_speed", "business_residential_code"]]

,brand_name,technology_name,max_advertised_download_speed,max_advertised_upload_speed,business_residential_code
336975,FirstLight Fiber,Fiber to the Premises,100000,100000,B
337133,FirstLight Fiber,Fiber to the Premises,100000,100000,B
299740,Xfinity,Fiber to the Premises,10000,10000,B
310812,GREEN MOUNTAIN ACCESS,Fiber to the Premises,10000,10000,X
316641,GREEN MOUNTAIN ACCESS,Fiber to the Premises,10000,10000,X


### 3.4 Business/residential code coverage
* `R` = residential only, `B` = business only, `X` = both.
* Cable has no `R` rows (only `B`/`X`); NGSO satellite (Starlink) is `X` everywhere.
* Because Starlink reports `X` at almost every location, **every one of the 282,377 locations enters the residential universe**. The residential universe is therefore defined as *locations with at least one R/X offer*. It can't separate homes from businesses on its own; that would require the licensed FCC Fabric.

In [9]:
pd.crosstab(df["technology_name"], df["business_residential_code"])

business_residential_code,B,R,X
technology_name,,,
Cable,905,0,190135
Copper,3781,95645,9224
Fiber to the Premises,4933,91910,107103
GSO Satellite,203881,282363,282340
LBR Fixed Wireless,1,24,7965
Licensed Fixed Wireless,64808,355011,143956
NGSO Satellite,0,0,282226
Unlicensed Fixed Wireless,0,91,8923


### 3.5 Low latency
GSO satellite is always `low_latency = 0`; every other technology is (almost) always 1.
The served/underserved tiers require low latency, as the FCC map and BEAD do.

In [10]:
df.groupby("technology_name")["low_latency"].mean().round(4).rename("share_low_latency").to_frame()

,share_low_latency
technology_name,
Cable,1.0000
Copper,0.9994
Fiber to the Premises,0.9993
GSO Satellite,0.0000
LBR Fixed Wireless,1.0000
Licensed Fixed Wireless,1.0000
NGSO Satellite,1.0000
Unlicensed Fixed Wireless,1.0000


## 4. Conclusion
The eight files are structurally clean and consistent. The issues found are semantic (B/R repeats, 0-Mbps offers, residential definition), and each is handled explicitly in `metrics.py` and documented in `docs/methodology.md`.